In [0]:
# ── DATA QUALITY FRAMEWORK ──────────────────────────────────
# Runs checks against the Silver layer and logs results
# to a dq_results table for monitoring and dashboarding

from pyspark.sql.functions import (
    col, count, when, isnan, isnull, 
    current_timestamp, lit
)
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType, TimestampType
import pandas as pd

# Read Silver layer
df = spark.table("silver_loan_applications")

print(f"Loaded {df.count()} rows from Silver layer")

In [0]:
# ── DQ CHECK ENGINE ──────────────────────────────────────────
# Each check returns a result dict with pass/fail and details

from datetime import datetime

results = []

def run_check(check_name, check_type, field, passed, total_rows, failed_rows, description):
    status = "PASS" if passed else "FAIL"
    results.append({
        "check_name": check_name,
        "check_type": check_type,
        "field": field,
        "status": status,
        "total_rows": total_rows,
        "failed_rows": failed_rows,
        "description": description,
        "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    emoji = "Passed" if passed else "Not passed"
    print(f"{emoji} [{status}] {check_name} — {failed_rows} issue(s) found")

total = df.count()

# ── COMPLETENESS CHECKS ──────────────────────────────────────
for field in ["application_id", "company_name", "province", "status"]:
    nulls = df.filter(col(field).isNull()).count()
    run_check(
        check_name=f"completeness_{field}",
        check_type="Completeness",
        field=field,
        passed=(nulls == 0),
        total_rows=total,
        failed_rows=nulls,
        description=f"{field} must not be null"
    )

# loan_amount null check (we know this will fail!)
loan_nulls = df.filter(col("loan_amount").isNull()).count()
run_check(
    check_name="completeness_loan_amount",
    check_type="Completeness",
    field="loan_amount",
    passed=(loan_nulls == 0),
    total_rows=total,
    failed_rows=loan_nulls,
    description="loan_amount must not be null"
)

# credit_score null check (we know this will fail too!)
score_nulls = df.filter(col("credit_score").isNull()).count()
run_check(
    check_name="completeness_credit_score",
    check_type="Completeness",
    field="credit_score",
    passed=(score_nulls == 0),
    total_rows=total,
    failed_rows=score_nulls,
    description="credit_score must not be null"
)

# ── CONSISTENCY CHECKS ──────────────────────────────────────
valid_provinces = ["AB","BC","MB","NB","NL","NS","NT","NU","ON","PE","QC","SK","YT"]
invalid_provinces = df.filter(~col("province").isin(valid_provinces)).count()
run_check(
    check_name="consistency_province",
    check_type="Consistency",
    field="province",
    passed=(invalid_provinces == 0),
    total_rows=total,
    failed_rows=invalid_provinces,
    description="province must be a valid Canadian province/territory code"
)

invalid_amounts = df.filter((col("loan_amount") <= 0)).count()
run_check(
    check_name="consistency_loan_amount",
    check_type="Consistency",
    field="loan_amount",
    passed=(invalid_amounts == 0),
    total_rows=total,
    failed_rows=invalid_amounts,
    description="loan_amount must be greater than 0"
)

invalid_scores = df.filter(
    (col("credit_score") < 300) | (col("credit_score") > 900)
).count()
run_check(
    check_name="consistency_credit_score",
    check_type="Consistency",
    field="credit_score",
    passed=(invalid_scores == 0),
    total_rows=total,
    failed_rows=invalid_scores,
    description="credit_score must be between 300 and 900"
)

# ── UNIQUENESS CHECK ─────────────────────────────────────────
total_ids = df.count()
distinct_ids = df.select("application_id").distinct().count()
duplicates = total_ids - distinct_ids
run_check(
    check_name="uniqueness_application_id",
    check_type="Uniqueness",
    field="application_id",
    passed=(duplicates == 0),
    total_rows=total,
    failed_rows=duplicates,
    description="application_id must be unique"
)

print(f"\n Ran {len(results)} checks")

In [0]:
# ── SAVE DQ RESULTS TO DELTA TABLE ──────────────────────────
# This becomes the source for our Power BI monitoring dashboard

dq_df = spark.createDataFrame(results)

dq_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dq_results")

print("DQ results saved successfully!")
print(f"\nSummary:")
dq_df.groupBy("status").count().show()
print("\nFailed checks:")
dq_df.filter(col("status") == "FAIL").show(truncate=False)